In [1]:
all_dataset = [
    "Cornell",
    "Texas",
    "Wisconsin",
    "reed98",
    "amherst41",
    "penn94",
    "Roman-empire",
    "cornell5",
    "Squirrel",
    "johnshopkins55",
    "Actor",
    "Minesweeper",
    "Questions",
    "Chameleon",
    "Tolokers",
    "Amazon-ratings",
    "genius",
    "pokec",
    "arxiv-year",
    "snap-patents",
    "ogbn-proteins",
    "Cora",
    "DBLP",
    "Computers",
    "PubMed",
    "Cora_ML",
    "SmallCora",
    "CS",
    "Photo",
    "Physics",
    "CiteSeer",
    "wiki",
    "Reddit"
]

In [2]:
import os
import sys
import math
import copy 
import torch 
import random
DEBUG = False
import numpy as np
import torch_sparse
import pandas as pd 
import pickle as pkl
from tqdm import tqdm
import time
import networkx as nx
from dgl import DGLGraph
from scipy import linalg
from pathlib import Path
from torch import Tensor
import scipy.sparse as sp
from random import randint
from dgl import transforms
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
from scipy import sparse, stats
from scipy.sparse import csgraph
from scipy.sparse import csr_matrix
from dgl import from_networkx, DGLGraph
from torch_geometric.utils import scatter
from dgl.data import citation_graph as citegrh
from torch_geometric.typing import SparseTensor
from torch_geometric.utils import to_undirected
from torch_geometric.utils import remove_self_loops
from sklearn.metrics.pairwise import cosine_similarity
# from ipynb.fs.full.Dataset import get_data_from_dataset
# from ipynb.fs.full.Dataset import get_data_from_dataset,train_val_test_mask
from ogb.nodeproppred import Evaluator, PygNodePropPredDataset
from typing import Callable, List, NamedTuple, Optional, Tuple, Union
from torch_geometric.utils import add_self_loops,add_remaining_self_loops
# from ipynb.fs.full.SpectralSparsifier import EffectiveResistance, LocalEffectiveResistance, get_sparse_adj_matrix

In [3]:
import DeviceDir

DIR, RESULTS_DIR = DeviceDir.get_directory()
device, NUM_PROCESSORS = DeviceDir.get_device()

In [59]:
from ipynb.fs.full.SGSLoadDataset import LOAD_DATASET

DATASET_NAME = "CiteSeer"
data, dataset  = LOAD_DATASET(DIR, DATASET_NAME)
num_classes = max(data.y).item()+1

CiteSeer N: 4230 E: 10674 F: 602 C: 6 d: 2.52 lr: 0.20 i: False s: False u: True


In [60]:
import os
import  scipy.sparse as sp
import numpy as np
import torch
import torch
import torch.nn as nn
import sys
import pickle as pkl
import numpy as np
import os
import torch.nn.functional as F
import os
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.sparse import coo_matrix
from torch_sparse import SparseTensor
import dgl.sparse as dglsp

In [61]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

_LAYER_UIDS = {}

def get_layer_uid(layer_name=''):
    """Helper function, assigns unique layer IDs."""
    if layer_name not in _LAYER_UIDS:
        _LAYER_UIDS[layer_name] = 1
        return 1
    else:
        _LAYER_UIDS[layer_name] += 1
        return _LAYER_UIDS[layer_name]

def sparse_dropout(x, rate, noise_shape):
    """
    Dropout for sparse tensors.
    """
    random_tensor = 1 - rate
    random_tensor += torch.rand(noise_shape, dtype=x.dtype, device=x.device)
    dropout_mask = torch.floor(random_tensor).bool()
    pre_out = x.coalesce()  # Ensure sparse tensor is in coalesced form
    retained_values = pre_out.values() * dropout_mask.float()
    return torch.sparse_coo_tensor(pre_out.indices(), retained_values * (1./(1 - rate)), pre_out.size())

def dot(x, y, sparse=False):
    """
    Wrapper for torch.matmul (sparse vs dense).
    """
    if sparse:
        res = torch.sparse.mm(x, y)
    else:
        res = torch.matmul(x, y)
    return res

class Dense(nn.Module):
    """Dense layer in PyTorch."""
    def __init__(self, input_dim, output_dim, dropout=0.0, act=F.relu, bias=False, activation=F.relu, featureless=False):
        super(Dense, self).__init__()
        self.act = act
        self.featureless = featureless
        self.dropout = dropout
        self.bias = bias

        # Weight initialization
        self.weights_ = nn.Parameter(torch.FloatTensor(input_dim, output_dim))
        nn.init.xavier_uniform_(self.weights_)

        if self.bias:
            self.bias_param = nn.Parameter(torch.FloatTensor(output_dim))
            nn.init.zeros_(self.bias_param)
        else:
            self.bias_param = None

    def forward(self, inputs):
        x = F.dropout(inputs, self.dropout, training=self.training)

        # transform
        output = torch.matmul(x, self.weights_)

        # bias
        if self.bias_param is not None:
            output += self.bias_param

        return self.act(output)


class GraphConvolution(nn.Module):
    """Graph convolution layer in PyTorch."""
    def __init__(self, input_dim, output_dim, dropout=0.0, is_sparse_inputs=False, activation=F.relu, bias=False, featureless=False):
        super(GraphConvolution, self).__init__()
        self.dropout = dropout
        self.activation = activation
        self.is_sparse_inputs = is_sparse_inputs
        self.featureless = featureless
        self.bias = bias

        # Weight initialization
        self.weights_ = nn.Parameter(torch.FloatTensor(input_dim, output_dim))
        nn.init.xavier_uniform_(self.weights_)

        if self.bias:
            self.bias_param = nn.Parameter(torch.FloatTensor(output_dim))
            nn.init.zeros_(self.bias_param)
        else:
            self.bias_param = None

    def forward(self, inputs,training):
        x, support_ = inputs

        # Apply dropout
        if training:
            x = F.dropout(x, self.dropout)

        # Convolve
        pre_sup = dot(x, self.weights_, sparse=self.is_sparse_inputs)
        output = dot(support_, pre_sup, sparse=False)

        # Bias
        if self.bias_param is not None:
            output += self.bias_param

        return self.activation(output)

In [62]:
def add_noisy_edge(shape, size, nb_noising_edges):
    noise_row = np.random.choice(range(size), nb_noising_edges)
    noise_col = np.random.choice(range(size), nb_noising_edges)
    noise_data = np.ones_like(noise_row)
    noise_adj = sp.coo_matrix((noise_data, (noise_row, noise_col)), shape=shape)
    return noise_adj

def preprocess_features(features):
    # Assuming features is a NumPy array, normalize if necessary
    return torch.tensor(features, dtype=torch.float32)

def sample_mask(idx, size):
    # Create a mask for training, validation, and testing
    mask = np.zeros(size, dtype=bool)
    mask[idx] = True
    return torch.tensor(mask, dtype=torch.bool)

def load_data(dataset_str):
    # Placeholder function, should load your dataset appropriately
    pass

In [63]:
def parse_index_file(filename):
    index = []
    with open(filename, 'r') as f:
        for line in f:
            index.append(int(line.strip()))
    return index

def sample_mask(idx, size):
    # Create a mask for training, validation, and testing
    mask = np.zeros(size, dtype=bool)
    mask[idx] = True
    return torch.tensor(mask, dtype=torch.bool)

def preprocess_features(features):
    return torch.tensor(features, dtype=torch.float32)

def load_data(dataset_str, directory):
    file_location = os.path.join(directory, "ind.{}.test.index".format(dataset_str))
    test_idx_reorder = parse_index_file(file_location)
    test_idx_range = np.sort(test_idx_reorder)
    names = ['x', 'y', 'tx', 'ty', 'allx', 'ally', 'graph']
    objects = []
    for name in names:
        file_path = os.path.join(directory, "ind.{}.{}".format(dataset_str, name))
        with open(file_path, 'rb') as f:
            if sys.version_info > (3, 0):
                objects.append(pkl.load(f, encoding='latin1'))
            else:
                objects.append(pkl.load(f))
    x, y, tx, ty, allx, ally, graph = tuple(objects)
    features = sp.vstack((allx, tx)).tolil()
    features[test_idx_reorder, :] = features[test_idx_range, :]

    labels = np.vstack((ally, ty))
    labels[test_idx_reorder, :] = labels[test_idx_range, :]

    adj = nx.adjacency_matrix(nx.from_dict_of_lists(graph))

    idx_test = test_idx_range.tolist()
    idx_train = range(140)
    idx_val = range(len(ally) - 500, len(ally))

    return x, y, tx, ty, allx, ally, adj, test_idx_range,features,labels,idx_train,idx_val,idx_test

In [64]:
import torch
import torch.nn.functional as F
from sklearn import metrics

def masked_softmax_cross_entropy(preds, labels, mask):
    """
    Softmax cross-entropy loss with masking.
    """
    loss = F.cross_entropy(preds, labels, reduction='none')  # No reduction to apply mask manually
    mask = mask.float()
    mask /= mask.mean()  # Normalize mask
    loss *= mask
    return loss.mean()

def masked_accuracy(preds, labels, mask):
    """
    Accuracy with masking.
    """
    correct_prediction = (preds.argmax(dim=1) == labels.argmax(dim=1)).float()
    mask = mask.float()
    mask /= mask.mean()  # Normalize mask
    correct_prediction *= mask
    return correct_prediction.mean()

def softmax_cross_entropy(preds, labels):
    """
    Softmax cross-entropy loss.
    """
    loss = F.cross_entropy(preds, labels)
    return loss

def sigmoid_cross_entropy(preds, labels):
    """
    Sigmoid cross-entropy loss.
    """
    labels = labels.float()
    loss = F.binary_cross_entropy_with_logits(preds, labels)
    return loss.mean()

def accuracy(preds, labels):
    """
    Accuracy.
    """
    correct_prediction = (preds.argmax(dim=1) == labels.argmax(dim=1)).float()
    return correct_prediction.mean()

def calc_f1(y_pred, y_true):
    """
    F1 score calculation.
    """
    y_pred = (y_pred > 0.5).float()  # Convert predictions to binary (0 or 1)
    y_true = y_true.float()
    return metrics.f1_score(y_true.cpu().numpy(), y_pred.cpu().numpy(), average="micro")

In [65]:
class GumbleGCN(nn.Module):
    def __init__(self, adj_matrix, shape, input_dim, output_dim, k, dropout=0.):
        super(GumbleGCN, self).__init__()

        self.adj_matrix = adj_matrix
        self.shape = shape
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.k = k
        self.hidden1 = 128
        self.hidden2 = 64
        self.weighted = True
        self.weight_decay = 0.0
        self.flag_value = 0 

        # Define layers
        self.layer1 = GraphConvolution(
            input_dim=self.input_dim,
            output_dim=self.hidden1,
            activation=F.relu,
            dropout=dropout,
            is_sparse_inputs=False
        )

        self.layer2 = GraphConvolution(
            input_dim=self.hidden1,
            output_dim=self.hidden2,
            activation=F.relu,
            dropout=dropout
        )

        self.layer3 = Dense(
            input_dim=self.hidden2,
            output_dim=self.output_dim,
            activation=lambda x: x,
            dropout=dropout
        )

        # Define additional dense layers for the sparse representation
        self.slayer1 = nn.Linear(2*self.input_dim+1, 32)
        self.slayer2 = nn.Linear(32, 1, bias=True)

    def sample_gumbel(self, shape, eps=1e-20):
        """Sample from Gumbel(0, 1)"""
        U = torch.rand(shape,dtype=torch.float32,device=self.adj_matrix.device)
        return -torch.log(-torch.log(U + eps) + eps)

    def gumbel_softmax_sample(self, logits, temperature, is_train):
        """Draw a sample from the Gumbel-Softmax distribution"""
        r = self.sample_gumbel(logits._values().shape)
        if is_train:
            values = torch.log(logits._values()) + r
        else:
            values = torch.log(logits._values())
        values /= temperature
        A = dglsp.spmatrix(self.adj_matrix._indices(), values, self.shape)
        A_softmax = dglsp.softmax(A,dim=1)
        y = torch.sparse.FloatTensor(A_softmax.indices(), A_softmax.val,self.shape)
        return y
    
    def forward(self, inputs, training=None):
        x, label, mask, temperature = inputs

        f1 = x[self.adj_matrix._indices()[0]]
        f2 = x[self.adj_matrix._indices()[1]]
        auv = self.adj_matrix._values().unsqueeze(-1)
        temp = torch.cat([f1, f2, auv], dim=-1)

        temp = F.relu(self.slayer1(temp))
        temp = self.slayer2(temp)
        z = temp.view(-1)

        A = dglsp.spmatrix(self.adj_matrix._indices(), z, self.shape)
        A_softmax = dglsp.softmax(A,dim=1)

        pi = torch.sparse.FloatTensor(A_softmax.indices(), A_softmax.val,self.shape)
        y = self.gumbel_softmax_sample(pi, temperature, training)

        y_dense = y.to_dense()
        top_k_v, top_k_i = torch.topk(y_dense, self.k, dim=-1)


        kth = torch.min(top_k_v, dim=-1)[0] + 1e-10
        kth = kth.unsqueeze(-1).expand_as(y_dense)
        mask2 = (y_dense >= kth).float()
        row_sum = mask2.sum(dim=-1)
        dense_support = mask2.float()

        if self.weighted:   #args.weighted
            dense_support *= y
        else:
            print("No gradient bug here!")
            exit()

        
        # Add self-loops
        self_edge = torch.eye(self.shape[0], device=y.device)
        dense_support = dense_support + self_edge

        # Normalize
        rowsum = dense_support.sum(dim=-1) + 1e-6  # Avoid NaN
        d_inv_sqrt = torch.pow(rowsum, -0.5)
        d_mat_inv_sqrt = torch.diag(d_inv_sqrt)
        ad = torch.matmul(dense_support, d_mat_inv_sqrt)
        ad_t = ad.transpose(0, 1)
        support = torch.matmul(ad_t, d_mat_inv_sqrt)

        # Pass through layers
        hidden = self.layer1((x, support), training)
        hidden = self.layer2((hidden, support), training)
        output = self.layer3(hidden)

        # Weight decay loss
        loss = 0
        for param in self.layer1.parameters():
            loss += self.weight_decay * torch.sum(param**2)

        # Cross-entropy loss
        loss += masked_softmax_cross_entropy(output, label, mask)
        # Accuracy
        acc = masked_accuracy(output, label, mask)
        if self.flag_value==0:
            # Calculate percentage of edges retained
            if torch.equal(self.adj_matrix._indices()[0], self.adj_matrix._indices()[1]):  
                total_edges = self.adj_matrix._nnz() // 2  
            else:
                total_edges = self.adj_matrix._nnz() 
            num_edges_retained = mask2.sum().item()
            print(f"The number of edges in the dataset: {total_edges}")
            print(f"The number of edges retained after sparsification: {num_edges_retained}")
            percentage_retained = (num_edges_retained / total_edges) * 100

            # Print the percentage of edges retained
            print(f"Percentage of edges retained after sparsification: {percentage_retained:.2f}%")
            self.flag_value=1

        return loss, acc

## Initialize the Model

In [66]:
data = data.to("cpu")


In [67]:
print(data.train_mask.sum(),data.val_mask.sum(),data.test_mask.sum())
print(data)

tensor(846) tensor(1692) tensor(1692)
Data(x=[4230, 602], edge_index=[2, 10674], y=[4230], train_mask=[4230], val_mask=[4230], test_mask=[4230])


In [68]:
num_classes = data.y.max().item() + 1
labels = torch.zeros(data.y.size(0), num_classes)
labels.scatter_(1, data.y.unsqueeze(1), 1)


y_train = np.zeros(labels.shape)
y_val = np.zeros(labels.shape)
y_test = np.zeros(labels.shape)

train_mask = data.train_mask
val_mask = data.val_mask
test_mask = data.test_mask

print(data.train_mask.sum(),val_mask.sum(),test_mask.sum())

y_train[data.train_mask, :] = labels[data.train_mask, :]
y_val[data.val_mask, :] = labels[data.val_mask, :]
y_test[data.test_mask, :] = labels[data.test_mask, :]

edge_index = data.edge_index
row, col = edge_index
indices = torch.stack([row, col], dim=0)
values = torch.ones(indices.size(1), dtype=torch.float32)
num_nodes = data.num_nodes
shape = torch.Size([num_nodes, num_nodes])
adj_tensor = torch.sparse_coo_tensor(indices, values, shape)
f1 = data.x[adj_tensor._indices()[0]]
f2 = data.x[adj_tensor._indices()[1]]

auv = adj_tensor._values().unsqueeze(-1)
temp = torch.cat([f1, f2, auv], dim=-1)

train_label = torch.tensor(y_train).to(device)
train_mask = train_mask.clone().detach().to(device)  # Fix for the warning
val_label = torch.tensor(y_val).to(device)
val_mask = val_mask.clone().detach().to(device)  # Fix for the warning
test_label = torch.tensor(y_test).to(device)
test_mask = test_mask.clone().detach().to(device)  # Fix for the warning
features = data.x.clone().detach().float().to(device)  # Fix for the warning and specify dtype
dropout = 0  # args.dropout
feature_tensor = data.x.clone().detach().float().to(device)

tensor(846) tensor(1692) tensor(1692)


In [69]:
data = data.to(device)
adj_tensor = adj_tensor.to(device)
f1 = f1.to(device)
f2 = f2.to(device)

In [78]:
k = 5
print("GumbleGCN Model Parameters")
print(f"Shape : {shape}")
print(f"Input Dim : {features.shape[-1]}")
print(f"Output Dim : {labels.shape[-1]}")
print(f"K : {k}")
print(f"Adjacency Sparse Tensor : {adj_tensor}")
model = GumbleGCN(adj_tensor, shape = shape, input_dim=features.shape[-1], output_dim=labels.shape[-1], k=k).to(device)

GumbleGCN Model Parameters
Shape : torch.Size([4230, 4230])
Input Dim : 602
Output Dim : 6
K : 5
Adjacency Sparse Tensor : tensor(indices=tensor([[   0,    0,    1,  ..., 4227, 4228, 4229],
                       [3514, 3617, 2951,  ..., 3808, 3551, 3949]]),
       values=tensor([1., 1., 1.,  ..., 1., 1., 1.]),
       device='cuda:0', size=(4230, 4230), nnz=10674, layout=torch.sparse_coo)


In [79]:
# From config 
temp_N = 50
temp_r = 1e-3
early_stopping = 100
# Optimizer setup
optimizer = torch.optim.Adam(model.parameters(), lr=0.01) # args.learning_rate

persist = 0
best_test_acc = 0
epochs =  500 #args.epochs
init_temp = 0.05

In [80]:
last_5_loss = []  # Initialize a list to store the last 5 losses

for epoch in range(epochs):
    if epoch % 50 == 0:
        decay_temp = np.exp(-1 * 1e-3 * epoch)
        temp = max(0.05, decay_temp)

    model.train()  # Set model to training mode
    optimizer.zero_grad()  # Clear previous gradients

    # Forward pass
    loss, acc = model((features, train_label, train_mask, temp))

    # Backward pass
    loss.backward()  # Compute gradients
    optimizer.step()  # Update weights

    # Append the current loss to the list and maintain only the last 5 losses
    last_5_loss.append(loss.item())
    if len(last_5_loss) > 5:
        last_5_loss.pop(0)

    # Check for convergence when at least 5 losses are available
    if len(last_5_loss) == 5 and np.std(last_5_loss) < 0.001:
        print(f"Convergence achieved at Epoch: {epoch}, Loss: {loss.item():.4f}, Temp: {temp:.4f}, Acc: {acc.item():.4f}")
        break

    # Print progress
    print(epoch, 'temp:', temp, 'loss:', loss.item(), 'acc:', acc.item())
    
#model.load_state_dict(torch.load('easy_checkpoint.pth'))
model.eval()
with torch.no_grad(
):
    test_loss, test_acc = model((features, test_label, test_mask, 1.0))
print(f'Test Loss: {test_loss.item()}, Test Acc: {test_acc.item()}')

The number of edges in the dataset: 10674
The number of edges retained after sparsification: 4101.98095703125
Percentage of edges retained after sparsification: 38.43%
0 temp: 1.0 loss: 1.7977349758148193 acc: 0.1288416087627411
1 temp: 1.0 loss: 1.7378242015838623 acc: 0.5271867513656616
2 temp: 1.0 loss: 1.638716220855713 acc: 0.7316784858703613
3 temp: 1.0 loss: 1.4934654235839844 acc: 0.8617021441459656
4 temp: 1.0 loss: 1.3246418237686157 acc: 0.9137115478515625
5 temp: 1.0 loss: 1.1401435136795044 acc: 0.9184396862983704
6 temp: 1.0 loss: 0.9458514451980591 acc: 0.924349844455719
7 temp: 1.0 loss: 0.755936324596405 acc: 0.936170220375061
8 temp: 1.0 loss: 0.5858334302902222 acc: 0.9385342597961426
9 temp: 1.0 loss: 0.44705888628959656 acc: 0.9444444179534912
10 temp: 1.0 loss: 0.3411214053630829 acc: 0.9503545761108398
11 temp: 1.0 loss: 0.26262637972831726 acc: 0.9550827145576477
12 temp: 1.0 loss: 0.2041044682264328 acc: 0.9609928727149963
13 temp: 1.0 loss: 0.1596212536096573 

In [81]:
import numpy as np

# Define the list
a = [0.890661, 0.8841607]

# Calculate mean and standard deviation
mean_a = np.mean(a)
std_a = np.std(a)

print("Mean:", mean_a)
print(f"Mean: {mean_a:.4f} +/- {std_a:.4f}")

Mean: 0.88741085
Mean: 0.8874 +/- 0.0033


In [74]:
EpochTimes = []
for epoch in range(epochs):
    
    start = time.time()
    
    if epoch % temp_N == 0:
        decay_temp = np.exp(-1 * temp_r * epoch)
        temp = max(0.05, decay_temp)

    model.train()
    optimizer.zero_grad()  # Zero out the gradients
    loss, acc = model((features, train_label, train_mask, temp)) 

    loss.backward()  
    optimizer.step() 
    
    EpochTimes.append(time.time() - start)

    if epoch%50 == 0:
        print(f'Train: Epoch {epoch}, Temp: {temp}, Training Loss: {loss.item()}, Training Accuracy: {acc.item()}')

    #if epoch % 25 == 0:
    model.eval()
    with torch.no_grad():  # Disable gradient computation for validation
        test_loss, test_acc = model((features, test_label, test_mask, 1.0))

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        persist = 0

    else:
        persist += 1

    if persist > early_stopping:
        break
    
    if epoch % 50 == 0:
        print(f'Test: Epoch {epoch}, Temp: {temp}, Loss: {loss.item()}, Acc: {acc.item()}, Val Loss: {test_loss.item()},  Val Acc: {test_acc.item()}')
    
print("Best test accuracy:", best_test_acc)
print("Mean epoch time: ", np.mean(EpochTimes))

Train: Epoch 0, Temp: 1.0, Training Loss: 0.013969921506941319, Training Accuracy: 0.9964538812637329
Test: Epoch 0, Temp: 1.0, Loss: 0.013969921506941319, Acc: 0.9964538812637329, Val Loss: 0.8376681208610535,  Val Acc: 0.8912529349327087
Train: Epoch 50, Temp: 0.951229424500714, Training Loss: 0.004673260264098644, Training Accuracy: 0.9976359009742737
Test: Epoch 50, Temp: 0.951229424500714, Loss: 0.004673260264098644, Acc: 0.9976359009742737, Val Loss: 1.1657581329345703,  Val Acc: 0.8782505989074707
Train: Epoch 100, Temp: 0.9048374180359595, Training Loss: 0.004426184576004744, Training Accuracy: 0.9976359009742737
Test: Epoch 100, Temp: 0.9048374180359595, Loss: 0.004426184576004744, Acc: 0.9976359009742737, Val Loss: 1.1321911811828613,  Val Acc: 0.8847517371177673
Best test accuracy: tensor(0.8913, device='cuda:0')
Mean epoch time:  0.005724345936494715


In [22]:
#model.load_state_dict(torch.load('easy_checkpoint.pth'))
model.eval()
with torch.no_grad(
):
    test_loss, test_acc = model((features, test_label, test_mask, 1.0))
print(f'Test Loss: {test_loss.item()}, Test Acc: {test_acc.item()}')

Test Loss: 2.7385237216949463, Test Acc: 0.512999951839447


In [18]:
print(best_test_acc)

0


In [19]:
#torch.save(model.state_dict(), 'easy_checkpoint.pth')  # Save model weights

Test Loss: 10.65407657623291, Test Acc: 0.5521235466003418


In [26]:
a= [0.78163254,0.7816325, 0.781632]
mean_a = np.mean(a)
std_dev_a = np.std(a)
print(mean_a, std_dev_a)

0.7816323466666667 2.456736769734297e-07
